In [46]:
# ============================================================
# NOTEBOOK 4 — SMOTENC OVERSAMPLING
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTENC

print("Libraries imported successfully.")

Libraries imported successfully.


In [47]:
# ============================================================
# CELL 2 — LOAD TRAINING AND TESTING DATA
# ============================================================

import pandas as pd
from pathlib import Path

# Project directories
PROCESSED_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")

# Load data saved from Notebook 2
X_train = pd.read_csv(
    PROCESSED_DIR / "X_train_clean.csv"
)

X_test = pd.read_csv(
    PROCESSED_DIR / "X_test_clean.csv"
)

y_train = pd.read_csv(
    PROCESSED_DIR / "y_train_clean.csv"
).squeeze()

y_test = pd.read_csv(
    PROCESSED_DIR / "y_test_clean.csv"
).squeeze()

print("Data loaded successfully.")

print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())
#=================Verify folders
#print("Processed directory:", PROCESSED_DIR.resolve())
#print("Files found:")

#for file in PROCESSED_DIR.iterdir():
#    print(file.name)

Data loaded successfully.

X_train shape: (455, 58)
y_train shape: (455,)
X_test shape: (114, 58)
y_test shape: (114,)

Training class distribution:
RISK_LABEL
0    329
1    126
Name: count, dtype: int64

Testing class distribution:
RISK_LABEL
0    83
1    31
Name: count, dtype: int64


In [48]:
# ============================================================
# CELL 3 — DEFINE NUMERICAL AND CATEGORICAL COLUMNS
# ============================================================
# The six continuous/numerical variables
numeric_columns = [
    "GPA_S1",
    "CA_AVG",
    "EXAM_AVG",
    "CLIN_AVG",
    "LAB_AVG",
    "ATT_RATE"
]

# All remaining predictors are categorical
categorical_columns = [
    col for col in X_train.columns
    if col not in numeric_columns
]
print("Number of numerical columns:", len(numeric_columns))
print("Number of categorical columns:", len(categorical_columns))

print("\nNumerical columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)

for i, col in enumerate(categorical_columns):
    print(i, col)

Number of numerical columns: 6
Number of categorical columns: 52

Numerical columns:
['GPA_S1', 'CA_AVG', 'EXAM_AVG', 'CLIN_AVG', 'LAB_AVG', 'ATT_RATE']

Categorical columns:
['ASSIGN_LATE', 'Age_group', 'Gender', 'Year_study', 'SES', 'Financial_diff', 'Employment_hrs', 'Study_hrs_day', 'Sleep_hrs', 'Self_risk_percep', 'Reviews notes within 24h', 'Understands content pre-exam', 'Seeks help when stuck', 'Uses library regularly', 'Completes readings', 'Takes organised notes', 'Concentration in self-study', 'Participates in class', 'Clinical takes study time', 'Prepared for clinical assess', 'Rotations affect performance', 'Adequate supervision', 'Schedule conflicts', 'Confident in clinical skills', 'Anxious about assessments', 'Sleep difficulty', 'Burnt out', 'Hopeless/unmotivated', 'Physical health affected', 'Considered break', 'Emotionally supported', 'Lecturers approachable', 'Sleep affects concentration', 'Regular exercise', 'Balanced diet', 'Health interferes studies', 'Takes rest 

In [49]:

# ============================================================
# CELL 4a — VERIFY COLUMN STRUCTURE
# ============================================================

unclassified = (
    set(X_train.columns)
    - set(numeric_columns)
    - set(categorical_columns)
)

overlap = (
    set(numeric_columns)
    & set(categorical_columns)
)

print("Total predictors:", X_train.shape[1])
print("Numerical:", len(numeric_columns))
print("Categorical:", len(categorical_columns))

print("\nUnclassified columns:")
print(unclassified)

print("\nColumns appearing in both lists:")
print(overlap)
# ============================================================
# CELL 4b — VERIFY COLUMN CONSISTENCY
# ============================================================

missing_categorical = [
    col for col in categorical_columns
    if col not in X_train.columns
]

missing_numerical = [
    col for col in numeric_columns
    if col not in X_train.columns
]

print("Missing categorical columns:", missing_categorical)
print("Missing numerical columns:", missing_numerical)

print(
    "\nTotal predictors:",
    len(categorical_columns) + len(numeric_columns)
)

Total predictors: 58
Numerical: 6
Categorical: 52

Unclassified columns:
set()

Columns appearing in both lists:
set()
Missing categorical columns: []
Missing numerical columns: []

Total predictors: 58


In [50]:
# ============================================================
# CELL 5 — GET CATEGORICAL COLUMN INDICES
# ============================================================

categorical_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_columns
]

print("Categorical column indices:")
print(categorical_indices)

print("\nNumber of categorical indices:")
print(len(categorical_indices))

# ============================================================
# CELL 6 — VERIFY NUMERICAL COLUMN INDICES
# ============================================================

numerical_indices = [
    X_train.columns.get_loc(col)
    for col in numeric_columns
]

print("Numerical columns:")
for col, idx in zip(numeric_columns, numerical_indices):
    print(idx, col)

print("\nNumber of numerical columns:", len(numerical_indices))

# ============================================================
# CELL 7 — CHECK DATA TYPES BEFORE SMOTENC
# ============================================================

print(X_train.dtypes.value_counts())

print("\nObject columns:")
print(X_train.select_dtypes(include="object").columns.tolist())

Categorical column indices:
[6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57]

Number of categorical indices:
52
Numerical columns:
0 GPA_S1
1 CA_AVG
2 EXAM_AVG
3 CLIN_AVG
4 LAB_AVG
5 ATT_RATE

Number of numerical columns: 6
int64      52
float64     6
Name: count, dtype: int64

Object columns:
[]


In [51]:
# ============================================================
# CELL 8 — CREATE SMOTENC
# ============================================================

smotenc = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy="auto",
    random_state=42,
    k_neighbors=5
)
print("SMOTENC configured successfully.")

# ============================================================
# CELL 9 — APPLY SMOTENC TO TRAINING DATA ONLY
# ============================================================

print("Applying SMOTENC...")

X_train_smotenc, y_train_smotenc = smotenc.fit_resample(
    X_train,
    y_train
)

print("SMOTENC completed successfully.")

SMOTENC configured successfully.
Applying SMOTENC...


SMOTENC completed successfully.


In [52]:
# ============================================================
# CELL 10 — CHECK RESAMPLED DATA --(Check the new dataset size)
# ============================================================

print("Original X_train shape:")
print(X_train.shape)

print("\nSMOTENC X_train shape:")
print(X_train_smotenc.shape)

print("\nOriginal y_train distribution:")
print(y_train.value_counts())

print("\nSMOTENC y_train distribution:")
print(y_train_smotenc.value_counts())

Original X_train shape:
(455, 58)

SMOTENC X_train shape:
(658, 58)

Original y_train distribution:
RISK_LABEL
0    329
1    126
Name: count, dtype: int64

SMOTENC y_train distribution:
RISK_LABEL
1    329
0    329
Name: count, dtype: int64


In [53]:
# ============================================================
# CELL 11 — RESTORE DATAFRAME STRUCTURE
# ============================================================

X_train_smotenc = pd.DataFrame(
    X_train_smotenc,
    columns=X_train.columns
)

y_train_smotenc = pd.Series(
    y_train_smotenc,
    name="RISK_LABEL"
)

print("X_train_smotenc:")
print(X_train_smotenc.shape)

print("\ny_train_smotenc:")
print(y_train_smotenc.shape)

X_train_smotenc:
(658, 58)

y_train_smotenc:
(658,)


In [54]:
# ============================================================
# CELL 12 — VALIDATE CATEGORICAL VALUES
# ============================================================

invalid_categories = {}

for col in categorical_columns:

    original_values = set(
        X_train[col].dropna().unique()
    )

    new_values = set(
        X_train_smotenc[col].dropna().unique()
    )

    invalid = new_values - original_values

    if invalid:
        invalid_categories[col] = sorted(invalid)

if invalid_categories:

    print("INVALID CATEGORICAL VALUES FOUND:")

    for col, values in invalid_categories.items():
        print(col, "→", values)

else:

    print("All SMOTENC categorical values are valid.")

All SMOTENC categorical values are valid.


In [55]:
# ============================================================
# CELL 13 — CHECK MISSING VALUES
# ============================================================

print(
    "Missing values in X_train_smotenc:",
    X_train_smotenc.isna().sum().sum()
)

print(
    "Missing values in y_train_smotenc:",
    y_train_smotenc.isna().sum()
)

# ============================================================
# CELL 14 — CHECK DUPLICATES
# ============================================================

print(
    "Duplicate rows in resampled X:",
    X_train_smotenc.duplicated().sum()
)

Missing values in X_train_smotenc:

 0
Missing values in y_train_smotenc: 0
Duplicate rows in resampled X: 0


In [56]:
# ============================================================
# CELL 15 — FINAL SMOTENC SUMMARY
# ============================================================

print("=" * 70)
print("SMOTENC SUMMARY")
print("=" * 70)

print("Original training samples:", len(X_train))
print("Original minority samples:", (y_train == 1).sum())
print("Original majority samples:", (y_train == 0).sum())

print("\nResampled training samples:", len(X_train_smotenc))
print("Resampled minority samples:", (y_train_smotenc == 1).sum())
print("Resampled majority samples:", (y_train_smotenc == 0).sum())

print("\nNumber of predictors:", X_train_smotenc.shape[1])
print("Numerical predictors:", len(numeric_columns))
print("Categorical predictors:", len(categorical_columns))

SMOTENC SUMMARY
Original training samples: 455
Original minority samples: 126
Original majority samples: 329

Resampled training samples: 658
Resampled minority samples: 329
Resampled majority samples: 329

Number of predictors: 58
Numerical predictors: 6
Categorical predictors: 52


In [57]:
# ============================================================
# CELL 16 — SAVE SMOTENC TRAINING DATA
# ============================================================

X_train_smotenc.to_csv(
    PROCESSED_DIR / "X_train_smotenc.csv",
    index=False
)

y_train_smotenc.to_csv(
    PROCESSED_DIR / "y_train_smotenc.csv",
    index=False
)

print("SMOTENC training data saved successfully.")

SMOTENC training data saved successfully.
